# NovaBank Digital Alert Triage With Noisy Outcomes

Review NovaBank Digital alerts where confirmed fraud is an **imperfect, noisy
label**, not a clean ground truth.

This notebook reuses the shared **Alert lifecycle** (the
`foundation_alert_lifecycle` Progressive data view established by the v0.1 /
v0.2 alert-governance module) and the supervised-baseline outputs from the
previous v0.4 slice. It uses the evaluation `limitation_summary` to explain why
confirmed-fraud cannot be treated as ground truth, and it never exposes
protected scenario answer keys in the triage flow.

**Glossary reminder:** a **Client** is the legal customer, a **User** is the
digital login identity, and the Alert lifecycle runs from suspicious activity
through alert, case, and case outcome.

In [ ]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

from banking_fraud_lab import (
    build_digital_banking_features,
    build_foundation_progressive_views,
    build_learner_facing_views,
    evaluate_alert_scores,
    generate_digital_fraud_scenarios_world,
)
from banking_fraud_lab.features import DIGITAL_BANKING_FEATURE_FAMILIES
from banking_fraud_lab.progressive_views import FOUNDATION_ALERT_LIFECYCLE
from banking_fraud_lab.schema import PROTECTED_SCENARIO_ANSWER_KEYS

pd.set_option("display.max_columns", 40)

## Generate Learner-Facing Data

The v0.4 digital generator deliberately injects noisy alert-triage outcomes: a
subset of cases close without a confirmation even though the underlying activity
was fraudulent, and others confirm fraud on activity that was actually benign.
Protected scenario answer keys hold the true labels but stay outside the
learner-facing views.

In [ ]:
tables = generate_digital_fraud_scenarios_world(
    seed=42,
    scale="small",
    scenario_prevalence=0.5,
    noisy_outcome_rate=0.3,
)
learner_tables = build_learner_facing_views(tables)

assert PROTECTED_SCENARIO_ANSWER_KEYS in tables
assert PROTECTED_SCENARIO_ANSWER_KEYS not in learner_tables
list(learner_tables)

## Reuse The Shared Alert Lifecycle

The `foundation_alert_lifecycle` Progressive data view exposes the Alert
lifecycle from suspicious activity through alert, case, and case outcome in one
traceable surface. It is the same view the alert-governance module uses, so the
digital triage flow stays aligned with the shared lifecycle instead of a
parallel structure.

In [ ]:
alert_lifecycle = build_foundation_progressive_views(learner_tables)[
    FOUNDATION_ALERT_LIFECYCLE.name
]
digital_lifecycle = alert_lifecycle[
    alert_lifecycle["institution_name"] == "NovaBank Digital"
].copy()
assert not digital_lifecycle.empty
assert PROTECTED_SCENARIO_ANSWER_KEYS not in digital_lifecycle.columns

digital_lifecycle[
    [
        "alert_id",
        "alert_type",
        "alert_status",
        "severity",
        "case_status",
        "outcome_type",
        "confirmed_fraud",
    ]
].head(10)

## Show That Confirmed-Fraud Is A Noisy Label

Confirmed-fraud is the operational triage decision, not a clean ground truth.
The digital data includes `unresolved` outcomes (cases closed without a
confirmation) alongside `confirmed-fraud` and `false-positive` outcomes. The mix
of outcome types is exactly why the `limitation_summary` warns against treating
confirmed-fraud as ground truth.

In [ ]:
outcome_summary = (
    digital_lifecycle.dropna(subset=["outcome_type"])
    .groupby("outcome_type", as_index=False)
    .agg(
        case_count=("case_id", "count"),
        confirmed_fraud_count=("confirmed_fraud", "sum"),
    )
)
assert "unresolved" in set(outcome_summary["outcome_type"])

# Confirmed-fraud only captures what the triage process confirmed, not every
# fraud case. Unresolved cases are closed without a confirmation, so confirmed
# counts understate fraud and cannot be treated as ground truth.
outcome_summary

## Reference The Supervised-Baseline Outputs

The supervised-baseline slice fits a small logistic-regression model on the `db_`
feature library and produces alert scores. We re-derive the baseline scores here
to drive the triage queue; the same model could be loaded from the
supervised-baseline notebook. Protected answer keys are never used to score
alerts.

In [ ]:
feature_frame = build_digital_banking_features(learner_tables)
triage_frame = (
    learner_tables["cases"][["case_id", "alert_id", "transaction_id"]]
    .merge(
        learner_tables["alerts"][["alert_id", "alert_type", "severity", "reason"]],
        on="alert_id",
        how="inner",
        validate="one_to_one",
    )
    .merge(
        learner_tables["case_outcomes"][["case_id", "confirmed_fraud", "outcome_type"]],
        on="case_id",
        how="inner",
        validate="one_to_one",
    )
    .merge(feature_frame, on="transaction_id", how="inner", validate="one_to_one")
)

feature_output_columns = [
    output_column
    for spec in DIGITAL_BANKING_FEATURE_FAMILIES
    for output_column in spec.output_columns
]
numeric_feature_columns = [
    column
    for column in feature_output_columns
    if pd.api.types.is_numeric_dtype(triage_frame[column])
]
preprocess = ColumnTransformer(
    [("numeric", StandardScaler(), numeric_feature_columns)],
    remainder="drop",
)
baseline_model = Pipeline(
    [
        ("preprocess", preprocess),
        (
            "model",
            LogisticRegression(
                random_state=42,
                solver="lbfgs",
                max_iter=1000,
                class_weight="balanced",
            ),
        ),
    ]
)
baseline_model.fit(
    triage_frame[numeric_feature_columns],
    triage_frame["confirmed_fraud"].astype(bool),
)
triage_frame = triage_frame.assign(
    score=baseline_model.predict_proba(triage_frame[numeric_feature_columns])[:, 1].round(6)
)
triage_frame[["alert_id", "alert_type", "outcome_type", "confirmed_fraud", "score"]].head(10)

## Build The Triage Queue With Capacity Limits

The triage queue ranks alerts by score against a fixed investigation capacity.
Because confirmed-fraud is noisy, the queue is an operational tool, not a
ranking of true fraud. High-scoring alerts still need human review precisely
because the label that trained the score is imperfect.

In [ ]:
alert_capacity = 6
triage_queue = triage_frame.sort_values("score", ascending=False).reset_index(drop=True)
triage_queue["queue_rank"] = triage_queue.index + 1
triage_queue["within_capacity"] = (triage_queue["queue_rank"] <= alert_capacity).astype(int)

queue_summary = pd.DataFrame(
    [
        {
            "alerts_ranked": len(triage_queue),
            "investigation_capacity": alert_capacity,
            "alerts_within_capacity": int(triage_queue["within_capacity"].sum()),
            "unresolved_within_capacity": int(
                triage_queue.loc[
                    triage_queue["within_capacity"].eq(1), "outcome_type"
                ].eq("unresolved").sum()
            ),
        }
    ]
)
queue_summary

## Explain Label Noise Through The Limitation Summary

The alert-aware evaluation reports a `limitation_summary` that keeps accuracy
out of scope and explains the operational reality. We evaluate the triage scores
through that same utility so the limitation wording is part of the learner-facing
output, not a footnote.

In [ ]:
report = evaluate_alert_scores(
    cases=triage_frame[["case_id", "alert_id"]],
    case_outcomes=learner_tables["case_outcomes"],
    alert_scores=triage_frame[["alert_id", "score"]],
    thresholds=(0.85, 0.60, 0.40, 0.25),
    alert_capacity=alert_capacity,
    investigation_cost_chf=200.0,
    false_positive_cost_chf=600.0,
    missed_fraud_cost_chf=40_000.0,
)

assert "accuracy is out of scope" in report["limitation_summary"]
assert PROTECTED_SCENARIO_ANSWER_KEYS not in report["limitation_summary"]

pd.DataFrame(
    [
        {
            "pr_auc": report["pr_auc"],
            "lowest_cost_threshold": report["lowest_cost_threshold"],
            "limitation_summary": report["limitation_summary"],
        }
    ]
)

## Triage Memo Template

Learners can adapt this memo structure when discussing a real digital alert
queue. It references the operational view (capacity, unresolved cases, cost) and
the limitation wording, never the protected answer keys.

In [ ]:
def triage_memo(row: pd.Series) -> str:
    """Return a one-line triage memo for a threshold summary row."""
    capacity_note = (
        "within current investigation capacity"
        if row["over_capacity_alerts"] == 0
        else f"over capacity by {int(row['over_capacity_alerts'])} alerts"
    )
    return (
        f"Threshold {row['threshold']:.2f}: precision {row['precision']:.2f}, "
        f"recall {row['recall']:.2f}, {int(row['alert_volume'])} alerts reviewed, "
        f"{capacity_note}. Confirmed-fraud is a noisy label; unresolved cases "
        "are reviewed alongside confirmed and false-positive outcomes."
    )


threshold_summaries = pd.DataFrame(report["threshold_summaries"])
triage_memos = pd.DataFrame(
    {"threshold": threshold_summaries["threshold"], "memo": threshold_summaries.apply(triage_memo, axis=1)}
)
triage_memos